# New Tokyo PoC Tester

## 1 · RT Remote Configurator

### Configuration details

import json
import socket

host='127.0.0.1'
port=35944

def send_configuration_and_receive_response(logname):
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
        print(f"Socket created. Sending configuration to {host}:{port}...")            
        configuration = {"data": []}

        configuration["data"].append({
            "type": "RT_CONFIGURATION_MESSAGE",
            "max_depth": 30,
            "max_num_paths_per_src": 10e10,
            "samples_per_src": 10e10,
            "los": True,
            "specular_reflection": True,
            "diffuse_reflection": True,
            "refraction": True,
            "diffraction": True,
            "corner_diffraction": True,
            "seed": 42,
            # Kalman filter parameters
            "use_kalman_filter": True,
            "kalman_process_var": 0.3, # 
            "kalman_meas_var": 0.8,    # Lower = trust measurement more
            "kalman_rt_var": 3,       # Higher = trust RT less
            # Monte Carlo parameters
            "montecarlo_realizations": 0,
            "montecarlo_max_position_jitter": 0,
            # Adaptive bias filter parameters
            "use_adaptive_bias_filter": False,
            "adaptive_bias_alpha_signal": 0.5,
            "adaptive_bias_alpha_bias": 0.6,
            "restart_log": False,
            "new_log_name": logname
        })

        message_bytes = json.dumps(configuration).encode('utf-8')
        sock.sendto(message_bytes, (host, port))

        print("Waiting for response...")
        response_data, server_address = sock.recvfrom(4096)
                    
        # Parse the response
        response_str = response_data.decode('utf-8')
        response = json.loads(response_str)
        print(f"Server Response: {response[0].get('response')}")

### Configuration Sender

In [ ]:
send_configuration_and_receive_response(logname="test_run")

## 2 · JSON Reader, Sender and Tester
FIXME: adapt to new PoC Scenario

In [ ]:
import json
import socket
import copy
import matplotlib.pyplot as plt
import numpy as np

def calculate_and_plot_results(timestamps, actual_rssi, estimated_rssi, raw_estimated_rssi, plot=False):
    
    if not timestamps:
        print("No data was collected to plot.")
        return
    
    if plot:
        fig_time, axes_time = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        fig_time.suptitle('RSSI Estimation vs Reality', fontsize=16)

    plot_keys = [
        ('RSSI_Vehicle1_2', 'RSSI 1 vs 2'),
        ('RSSI_Vehicle1_3', 'RSSI 1 vs 3'),
        ('RSSI_Vehicle2_3', 'RSSI 2 vs 3')
    ]

    print("\n" + "="*81)
    print(f"{'LINK ID':<13} | {'RMSE':<8} | {'MAE':<8} | {'BIAS':<8} | {'95th %':<8} | {'PEARSON':<8} | {'RAW BIAS':<8} |")
    print("-" * 80)

    for i, (key, title) in enumerate(plot_keys):
        act = np.array(actual_rssi[key])
        est = np.array(estimated_rssi[key])
        raw = np.array(raw_estimated_rssi[key])
                    
        # Filter out "0" values
        mask = act != 0
        valid_act = act[mask]
        valid_est = est[mask]
        valid_raw = raw[mask]
        
        if len(valid_act) == 0:
            print(f"{title:<13} |   - - - No field measurement for comparison - - -")
            if plot:
                ax_t = axes_time[i]
                #ax_t.plot(timestamps, act, 'b-o', label='Actual', markersize=4, alpha=0.6)
                ax_t.plot(timestamps, est, 'r--x', label='Estimated', markersize=4, alpha=0.6)
                ax_t.plot(timestamps, raw, 'g-.s', label='Raw Estimated', markersize=4, alpha=0.6)
                ax_t.set_ylabel('RSSI (dBm)')
                ax_t.set_title(title)
                ax_t.legend(loc='lower right')
                ax_t.grid(True, linestyle='--', alpha=0.5)
            continue

        # --- 2. Calculate New Metrics ---
        errors = valid_est - valid_act                      # Signed error
        abs_errors = np.abs(errors)                         # Absolute error
        raw_errors = valid_raw - valid_act                  # Signed error for raw
        
        rmse = np.sqrt(np.mean(errors**2))                  # RMSE (Root Mean Square Error)
        mae = np.mean(abs_errors)                           # MAE (Mean Absolute Error)
        bias = np.mean(errors)                              # Bias
        raw_bias = np.mean(raw_errors)                      # Bias for raw
        p95 = np.percentile(abs_errors, 95)                # 95th Percentile of Absolute Error
        corr = np.corrcoef(valid_act, valid_est)[0, 1]     # Pearson Correlation Coefficient

        # Store statistics
        statistics = {
            'link': title,
            'rmse': rmse,
            'mae': mae,
            'bias': bias,
            'p95': p95,
            'pearson_corr': corr,
            'raw_bias': raw_bias
        }

        # Print Row in Table
        print(f"{title:<13} | {rmse:>6.2f}dB | {mae:>6.2f}dB | {bias:>6.2f}dB | {p95:>6.2f}dB |   {corr:>6.2f} | {raw_bias:>6.2f}dB |")
        if plot:
            ax_t = axes_time[i]
            ax_t.plot(timestamps, act, 'b-o', label='Measured', markersize=4, alpha=0.6)
            ax_t.plot(timestamps, est, 'r--x', label='Simulated Ground Truth', markersize=4, alpha=0.6)
            #ax_t.plot(timestamps, raw, 'g-.s', label='Raw Estimated', markersize=4, alpha=0.6)
            ax_t.set_ylabel('RSSI [dBm]')
            ax_t.set_title(title)
            ax_t.legend(loc='lower right')
            ax_t.grid(True, linestyle='--', alpha=0.5)

    print("="*81)
    if plot:
        axes_time[-1].set_xlabel('Time (s)')
        plt.tight_layout()
        plt.show()
    
    return statistics

def send_data_in_pairs(input_file, host='127.0.0.1', port=35944, verbose=False, error_jitter=0.0):
    """
    Reads a merged JSON file, sends pairs of elements over a UDP socket,
    compares the response with actual data, and plots the results.

    Args:
        input_file (str): The file path for the merged JSON data.
        host (str): The hostname or IP address to send data to.
        port (int): The port number to use for the UDP socket.
        verbose (bool): If True, prints detailed logs.
        error_jitter (float): Maximum position jitter to inject (in meters).
    """
    try:
        with open(input_file, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Input file not found at {input_file}")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {input_file}")
        return

    # Lists to store data for final plotting and analysis
    timestamps = []
    actual_rssi = {'RSSI_Vehicle1_2': [], 'RSSI_Vehicle1_3': [], 'RSSI_Vehicle2_3': []}
    estimated_rssi = {'RSSI_Vehicle1_2': [], 'RSSI_Vehicle1_3': [], 'RSSI_Vehicle2_3': []}
    raw_estimated_rssi = {'RSSI_Vehicle1_2': [], 'RSSI_Vehicle1_3': [], 'RSSI_Vehicle2_3': []}

    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
        print(f"Socket created. Sending data to {host}:{port}")

        for i in range(len(data) - 1):
            element_1 = data[i]
            element_1_modified = copy.deepcopy(element_1)

            p_feedback = 1
            
            # Add probability to RSSI measurement feedback to Digital Twin
            element_1_modified['data']['RSSI_Vehicle1_2'] = 0 if np.random.uniform(0, 1) > p_feedback else element_1_modified['data']['RSSI_Vehicle1_2']
            element_1_modified['data']['RSSI_Vehicle1_3'] = 0 if np.random.uniform(0, 1) > p_feedback else element_1_modified['data']['RSSI_Vehicle1_3']
            element_1_modified['data']['RSSI_Vehicle2_3'] = 0 if np.random.uniform(0, 1) > p_feedback else element_1_modified['data']['RSSI_Vehicle2_3']
            
            element_2_original = data[i+1]
            element_2_modified = copy.deepcopy(element_2_original)

            # Set the RSSI values of the second element to 0
            if 'data' in element_2_modified:
                element_2_modified['data']['RSSI_Vehicle1_2'] = 0
                element_2_modified['data']['RSSI_Vehicle1_3'] = 0
                element_2_modified['data']['RSSI_Vehicle2_3'] = 0
                
                for car in ['x1', 'x2', 'x3']:
                    if car in element_2_modified['data']:
                        current_car = element_2_modified['data'][car]
                        oracle_x = current_car['x']
                        oracle_z = current_car['z']

                        if car == 'x2':
                            new_x = oracle_x + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            new_z = oracle_z + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            element_2_modified['data'][car]['x'] = new_x
                            element_2_modified['data'][car]['z'] = new_z

                        elif car == 'x1':
                            new_x = -13467.166 + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            new_z = -43713.134 + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            element_2_modified['data'][car]['x'] = new_x
                            element_2_modified['data'][car]['z'] = new_z

                        elif car == 'x3':
                            new_x = -13491.5322 + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            new_z = -43709.6992 + np.random.uniform(error_jitter/3, error_jitter/2)*(-1 if np.random.rand() > 0.5 else 1)
                            element_2_modified['data'][car]['x'] = new_x
                            element_2_modified['data'][car]['z'] = new_z
                    
                        if verbose:
                            print(f"Inaccuracy injected for {car}: x={(oracle_x - new_x):.2f}, z={(oracle_z - new_z):.2f}")
            
            message_payload = [element_1, element_2_modified]
            
            try:
                message_bytes = json.dumps(message_payload).encode('utf-8')
                if verbose:
                    print(f"\n--- Sending elements {i+1} and {i+2} (clock: {element_2_original['clock']:.2f}s) ---")
                sock.sendto(message_bytes, (host, port))

                if verbose:
                    print("Waiting for response...")
                response_data, server_address = sock.recvfrom(4096)
                
                # Parse the response
                response_str = response_data.decode('utf-8')
                if verbose:
                    print(f"Received response: {response_str}")
                estimated_data = json.loads(response_str)
                
                if estimated_data and isinstance(estimated_data, list):
                    estimated_values = estimated_data[0]
                    actual_values = element_2_original['data']

                    # Store data for plotting
                    timestamps.append(element_2_original['clock'])
                    
                    est_1_2 = int(estimated_values['data'].get('RSSI_Vehicle1_2', 0))
                    act_1_2 = actual_values.get('RSSI_Vehicle1_2', 0)
                    raw_rssi_1_2 = estimated_values['data'].get('raw_RSSI_Vehicle1_2', 0)
                    actual_rssi['RSSI_Vehicle1_2'].append(act_1_2)
                    estimated_rssi['RSSI_Vehicle1_2'].append(est_1_2)
                    raw_estimated_rssi['RSSI_Vehicle1_2'].append(raw_rssi_1_2)
                    
                    est_1_3 = int(estimated_values['data'].get('RSSI_Vehicle1_3', 0))
                    act_1_3 = actual_values.get('RSSI_Vehicle1_3', 0)
                    raw_rssi_1_3 = estimated_values['data'].get('raw_RSSI_Vehicle1_3', 0)
                    actual_rssi['RSSI_Vehicle1_3'].append(act_1_3)
                    estimated_rssi['RSSI_Vehicle1_3'].append(est_1_3)
                    raw_estimated_rssi['RSSI_Vehicle1_3'].append(raw_rssi_1_3)

                    est_2_3 = int(estimated_values['data'].get('RSSI_Vehicle2_3', 0))
                    act_2_3 = actual_values.get('RSSI_Vehicle2_3', 0)
                    raw_rssi_2_3 = estimated_values['data'].get('raw_RSSI_Vehicle2_3', 0)
                    actual_rssi['RSSI_Vehicle2_3'].append(act_2_3)
                    estimated_rssi['RSSI_Vehicle2_3'].append(est_2_3)
                    raw_estimated_rssi['RSSI_Vehicle2_3'].append(raw_rssi_2_3)

                    # Print comparison for this step
                    if verbose:
                        print("Comparison:")
                        print(f"  RSSI 1-2 | Actual: {act_1_2:6.2f}, Estimated: {est_1_2:6.2f}, Diff: {abs(act_1_2 - est_1_2):.2f}")
                        print(f"  RSSI 1-3 | Actual: {act_1_3:6.2f}, Estimated: {est_1_3:6.2f}, Diff: {abs(act_1_3 - est_1_3):.2f}")
                        print(f"  RSSI 2-3 | Actual: {act_2_3:6.2f}, Estimated: {est_2_3:6.2f}, Diff: {abs(act_2_3 - est_2_3):.2f}")
            except json.JSONDecodeError:
                print(f"Error: Could not decode the received JSON response: {response_str}")
            except socket.error as e:
                print(f"A socket error occurred: {e}")
                break
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
                break
            
    # After the loop, calculate and plot the results
    statistics = calculate_and_plot_results(timestamps, actual_rssi, estimated_rssi, raw_estimated_rssi, plot=True) # Disable plotting
    return statistics


if __name__ == '__main__':
    merged_json_file = 'log_1759489893_nobf_sidelink_first.json'
    #merged_json_file = 'smoothed_trajectory.json'
    #merged_json_file = 'log_1759488416_sidelink_first_try.json' # Beamforming!
    statistics_all = {}
    
    for jitter in list(np.arange(0, 1, 1)): # Start, End, Step
        print(f"\n=== Running test with error_max_jitter = {jitter} ===")
        #send_configuration_and_receive_response(logname=f"test_{jitter:.2f}")
        statistics_all[jitter] = send_data_in_pairs(merged_json_file, verbose=True, error_jitter=jitter)